# #7DaysOfCode — Agentes de IA & Engenharia de Prompts
## Resolução Técnica — Dia 6: Motor Agentic RAG com `ChromaDB` e `gemini-embedding-001`

Este notebook apresenta a implementação técnica do **Dia 6**, construindo o pipeline de **RAG (*Retrieval-Augmented Generation*)** sobre o manual normativo [`politicas_troca.txt`](../data/politicas_troca.txt) e expondo a busca vetorial como ferramenta (`@tool`) para a assistente **Clara**.

### Escopo Técnico desta Implementação
- Carregamento e fatiamento semântico (*chunking*) do arquivo `data/politicas_troca.txt` com `RecursiveCharacterTextSplitter` (`chunk_size=500`, `chunk_overlap=80`) e metadados por seção;
- Geração de Embeddings com `GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")` e indexação vetorial no **ChromaDB**;
- Criação da ferramenta `@tool consultar_manual_politicas_rag` acoplada ao motor ReAct para consulta sob demanda e citação explícita das seções do manual.

---

## Etapa 1: Preparação do Ambiente, Dependências de RAG e Conexão Segura

Instalamos os pacotes da stack (`langchain`, `langchain-core`, `langchain-google-genai`, `langchain-chroma`, `langchain-text-splitters`, `chromadb`, `pydantic` e `python-dotenv`) e configuramos a `GOOGLE_API_KEY`.

In [ ]:
# Nota: No Google Colab, pode aparecer um aviso sobre versão do 'google-auth', mas a instalação funciona normalmente!
!pip install -q -U langchain langchain-core langchain-google-genai langchain-chroma langchain-community langchain-text-splitters chromadb pydantic python-dotenv

In [ ]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("GOOGLE_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
        print("Chave GOOGLE_API_KEY carregada com sucesso dos Secrets do Google Colab!")
    except Exception:
        os.environ["GOOGLE_API_KEY"] = getpass.getpass("Digite sua GOOGLE_API_KEY (Google AI Studio): ")
        print("Chave configurada com sucesso para esta sessão!")
else:
    print("GOOGLE_API_KEY já está configurada no ambiente!")

---
## Etapa 2: Carregamento e Fatiamento Semântico (*Chunking*) do Manual `politicas_troca.txt`

Carregamos o arquivo [`../data/politicas_troca.txt`](../data/politicas_troca.txt) e o dividimos em blocos menores (*chunks*) para indexação vetorial.

### Configuração do `RecursiveCharacterTextSplitter` (`chunk_size=500`, `chunk_overlap=80`)
- Utilizamos separadores hierárquicos (`["\n## ", "\n\n", "\n", " "]`) para respeitar a divisão de seções (`## `) do manual;
- Com `chunk_size=500` e `chunk_overlap=80`, mantemos cada cláusula normativa acompanhada de seu contexto e enriquecemos cada objeto `Document` com metadados (`secao` e `fonte`).

In [ ]:
from pathlib import Path
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

def localizar_arquivo_dados(nome_arquivo: str) -> Path:
    """Localiza o arquivo na pasta '../data/', 'data/' ou no diretório atual do Colab."""
    for candidato in [Path(f"../data/{nome_arquivo}"), Path(f"data/{nome_arquivo}"), Path(nome_arquivo)]:
        if candidato.exists():
            return candidato
    raise FileNotFoundError(
        f"Arquivo '{nome_arquivo}' não encontrado! Certifique-se de que ele está em '../data/{nome_arquivo}' "
        f"ou faça o upload de '{nome_arquivo}' na aba de Arquivos do Google Colab."
    )

# Carrega diretamente o arquivo oficial '../data/politicas_troca.txt'
caminho_arquivo_politicas = localizar_arquivo_dados("politicas_troca.txt")
texto_manual_politicas = caminho_arquivo_politicas.read_text(encoding="utf-8")

# Configurando o RecursiveCharacterTextSplitter conforme especificado no desafio
splitter_politicas = RecursiveCharacterTextSplitter(
    separators=["\n## ", "\n\n", "\n", " "],
    chunk_size=500,
    chunk_overlap=80,
    keep_separator=True
)

# Fatiando o manual e preservando o título da seção em cada chunk para que nenhuma cláusula perca seu cabeçalho
secoes_brutas = [s.strip() for s in texto_manual_politicas.split("\n## ") if s.strip()]
documentos_politicas: list[Document] = []

for secao in secoes_brutas:
    if secao.startswith("# ====================================================================="):
        continue
    texto_secao = secao if secao.startswith("## ") else f"## {secao}"
    titulo_secao = texto_secao.splitlines()[0].replace("## ", "").strip()

    sub_chunks = splitter_politicas.split_text(texto_secao)
    for idx_sub, chunk in enumerate(sub_chunks, 1):
        # Garante que mesmo sub-chunks mantenham a referência explícita do título da seção
        conteudo_chunk = chunk if chunk.startswith("## ") else f"## {titulo_secao}\n{chunk}"
        documentos_politicas.append(
            Document(
                page_content=conteudo_chunk.strip(),
                metadata={
                    "fonte": caminho_arquivo_politicas.name,
                    "secao": titulo_secao,
                    "fragmento": idx_sub
                }
            )
        )

print(f"Arquivo carregado: {caminho_arquivo_politicas.resolve()}")
print(f"Total de chunks semânticos gerados: {len(documentos_politicas)}\n")
for i, doc in enumerate(documentos_politicas, 1):
    print(f"[Chunk #{i:02d} | {len(doc.page_content):03d} chars | Seção: {doc.metadata['secao']}]")
    print(doc.page_content)
    print("-" * 90)

---
## Etapa 3: Geração de Embeddings (`gemini-embedding-001`) e Indexação no ChromaDB

Transformamos os documentos fatiados em vetores numéricos (*Embeddings*) utilizando o modelo `models/gemini-embedding-001` e os armazenamos no **ChromaDB**:

```text
┌──────────────────────────┐     ┌─────────────────────────────┐     ┌──────────────────────────┐
│ Chunks do Manual Oficial │ --> │ GoogleGenerativeAIEmbeddings│ --> │   Vector Store ChromaDB  │
│  (politicas_troca.txt)   │     │ (models/gemini-embedding-001│     │ (retriever top-k = 2)    │
└──────────────────────────┘     └─────────────────────────────┘     └──────────────────────────┘
```

Em seguida, instanciamos o `retriever_politicas = vectorstore_politicas.as_retriever(search_kwargs={"k": 2})` para recuperar os 2 fragmentos semanticamente mais relevantes para cada consulta.

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

# 1. Instanciando o modelo oficial de Embeddings do Google Gemini
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

# 2. Criando o banco vetorial em memória no ChromaDB com os chunks do manual
vectorstore_politicas = Chroma.from_documents(
    documents=documentos_politicas,
    embedding=embeddings,
    collection_name="manual_politicas_novamart_dia6"
)

# 3. Configurando o Retriever para buscar os top-2 trechos mais relevantes (k=2)
retriever_politicas = vectorstore_politicas.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2}
)

# Teste rápido de sanidade do motor vetorial
vetor_amostra = embeddings.embed_query("Qual a validade do código de postagem e-ticket dos Correios?")
docs_teste = retriever_politicas.invoke("Qual a validade do código de postagem e-ticket dos Correios?")

print(f"Modelo de Embeddings ativo: models/gemini-embedding-001 (Dimensão do vetor: {len(vetor_amostra)})")
print(f"ChromaDB indexado com sucesso! Top-{len(docs_teste)} seções recuperadas no teste rápido:")
for idx, d in enumerate(docs_teste, 1):
    print(f"  {idx}. Seção recuperada: {d.metadata['secao']}")

---
## Etapa 4: Encapsulando o Retriever na `@tool` (`consultar_manual_politicas_rag`)

No padrão **Agentic RAG**, expomos o `retriever` como uma ferramenta (`@tool`) para que o agente acione a busca vetorial sob demanda:
1. A ferramenta `@tool def consultar_manual_politicas_rag(duvida_politica: str) -> dict:` consulta o ChromaDB e retorna os 2 trechos mais relevantes com seus metadados de seção;
2. O motor `consultar_clara_rag_dia6` instrui a agente a basear sua resposta exclusivamente nas cláusulas recuperadas e a citar a seção oficial correspondente.

In [ ]:
import json
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage

# Inicializando o LLM da Clara (gemini-3.8-flash com temperature=0.0 para máxima fidelidade factual)
llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    temperature=0.0
)

@tool
def consultar_manual_politicas_rag(duvida_politica: str) -> dict:
    """
    Realiza uma busca semântica (RAG) no Manual Oficial de Políticas de Trocas, Devoluções e Reembolsos
    da NovaMart ('politicas_troca.txt') indexado no ChromaDB.
    Use esta ferramenta SEMPRE que o cliente fizer perguntas sobre:
    - Validade e renovação de código de postagem dos Correios (e-ticket / logística reversa);
    - Prazos de conferência no CD e prazos de reembolso/estorno por PIX, Boleto Bancário, Cartão ou Vale-Troca;
    - Regras de 1ª troca gratuita vs. cobrança de frete a partir da 2ª troca do mesmo pedido;
    - Prazos de reporte de avaria de transporte (48h) ou análise técnica de defeito de fabricação (15 dias úteis);
    - Exceções e categorias com restrição de troca/devolução (gift cards, softwares digitais, cosméticos, moda íntima, perecíveis).
    """
    docs_recuperados = retriever_politicas.invoke(duvida_politica)
    trechos_formatados = []
    for rank, doc in enumerate(docs_recuperados, 1):
        trechos_formatados.append({
            "rank_relevancia": rank,
            "fonte": doc.metadata.get("fonte", "politicas_troca.txt"),
            "secao": doc.metadata.get("secao", "Seção Geral"),
            "conteudo_oficial": doc.page_content
        })
    return {
        "consulta_semantica": duvida_politica,
        "total_trechos_recuperados": len(trechos_formatados),
        "trechos_manual_oficial": trechos_formatados
    }

ferramentas_rag_dia6 = [consultar_manual_politicas_rag]
mapa_ferramentas_rag_dia6 = {f.name: f for f in ferramentas_rag_dia6}
llm_com_rag_dia6 = llm.bind_tools(ferramentas_rag_dia6)

SYSTEM_PROMPT_RAG_DIA6 = """
Você é a Clara, Consultora Sênior de Políticas e Logística Reversa da NovaMart, operando no paradigma Agentic RAG (ReAct + Retrieval-Augmented Generation).

<diretrizes_rag_obrigatorias>
1. SEMPRE que o cliente fizer qualquer pergunta sobre regras de troca, devolução, prazos de conferência e reembolso (PIX, Boleto, Cartão, Vale-Troca), validade/renovação de código dos Correios (e-ticket), custos de frete em uma 2ª troca, avaria no transporte ou itens com restrição (como gift cards, cosméticos, roupas íntimas), você DEVE obrigatoriamente acionar a ferramenta 'consultar_manual_politicas_rag' ANTES de responder.
2. FIDELIDADE ESTRITA À FONTE (GROUNDING): Responda baseando-se exclusivamente nas cláusulas retornadas pela ferramenta 'consultar_manual_politicas_rag'. Nunca invente prazos, taxas ou exceções.
3. CITAÇÃO OBRIGATÓRIA DE SEÇÃO: Em toda resposta ao cliente, cite explicitamente o número da Seção e Subseção oficial do Manual de Políticas da NovaMart (ex: 'Conforme a Seção 6 (itens 6.1 e 6.2) do nosso Manual de Políticas...', 'De acordo com a Seção 5 (item 5.1)...', 'Segundo a Seção 2.2...', 'Conforme a Seção 4.4...').
4. CLAREZA EM CÁLCULOS DE PRAZO: Quando a dúvida envolver mais de uma etapa (ex: prazo de conferência de qualidade no CD + prazo de depósito bancário), detalhe cada etapa citada no manual e apresente também a soma total em dias úteis para facilitar o entendimento do cliente.
</diretrizes_rag_obrigatorias>

<protocolo_react_obrigatorio>
- ANTES de chamar a ferramenta 'consultar_manual_politicas_rag', escreva um parágrafo curto iniciado por 'PENSAMENTO:' explicando qual dúvida jurídica/operacional você precisa verificar no manual.
- Após receber a OBSERVAÇÃO do ChromaDB com os trechos oficiais, escreva a resposta final completa, acolhedora e fundamentada nas seções recuperadas (sem o prefixo 'PENSAMENTO:').
</protocolo_react_obrigatorio>
"""

def _extrair_texto_limpo(conteudo) -> str:
    """Extrai o texto limpo de AIMessage.content (seja string ou lista de blocos do Gemini)."""
    if isinstance(conteudo, str):
        return conteudo.strip()
    if isinstance(conteudo, list):
        partes = []
        for bloco in conteudo:
            if isinstance(bloco, dict) and bloco.get("type") == "text":
                partes.append(bloco.get("text", ""))
            elif isinstance(bloco, str):
                partes.append(bloco)
        return "\n".join(partes).strip()
    return str(conteudo).strip()

def consultar_clara_rag_dia6(pergunta_cliente: str, max_iter: int = 4, verbose: bool = True) -> dict:
    """
    Executa o ciclo Agentic RAG da Clara sobre o manual 'politicas_troca.txt' no ChromaDB.
    """
    historico = [
        SystemMessage(content=SYSTEM_PROMPT_RAG_DIA6),
        HumanMessage(content=pergunta_cliente)
    ]
    trilha_rag = []

    if verbose:
        print(f"🗣️ [PERGUNTA DO CLIENTE]: {pergunta_cliente}")
        print("-" * 95)

    for iteracao in range(1, max_iter + 1):
        resposta_ia = llm_com_rag_dia6.invoke(historico)
        historico.append(resposta_ia)
        texto_ia = _extrair_texto_limpo(resposta_ia.content)

        if not resposta_ia.tool_calls:
            if verbose:
                print(f"🤖 [RESPOSTA FUNDAMENTADA DA CLARA (Iteração {iteracao})]:")
                print(texto_ia)
            return {
                "pergunta": pergunta_cliente,
                "iteracoes": iteracao,
                "trilha_rag": trilha_rag,
                "resposta_final": texto_ia
            }

        pensamento = texto_ia if texto_ia else "PENSAMENTO: Consultando o Manual de Políticas da NovaMart no ChromaDB para verificar a cláusula exata."
        if verbose:
            print(f"🧠 [ITERAÇÃO {iteracao} — PENSAMENTO (Thought)]:")
            print(f"   {pensamento}")

        for chamada in resposta_ia.tool_calls:
            nome_tool = chamada["name"]
            args_tool = chamada["args"]
            if verbose:
                print(f"🔍 [ITERAÇÃO {iteracao} — AÇÃO RAG (Action)]:")
                print(f"   -> Tool: {nome_tool}({json.dumps(args_tool, ensure_ascii=False)})")

            resultado_tool = mapa_ferramentas_rag_dia6[nome_tool].invoke(args_tool)
            secoes_encontradas = [t["secao"] for t in resultado_tool.get("trechos_manual_oficial", [])]

            if verbose:
                print(f"📚 [ITERAÇÃO {iteracao} — OBSERVAÇÃO VETORIAL (ChromaDB Top-2)]:")
                print(f"   <- Seções recuperadas: {secoes_encontradas}")

            trilha_rag.append({
                "iteracao": iteracao,
                "pensamento": pensamento,
                "argumentos_busca": args_tool,
                "secoes_recuperadas": secoes_encontradas,
                "retorno_completo": resultado_tool
            })

            historico.append(
                ToolMessage(
                    content=json.dumps(resultado_tool, ensure_ascii=False),
                    tool_call_id=chamada["id"],
                    name=nome_tool
                )
            )

    resposta_fechamento = llm.invoke(historico)
    texto_final = _extrair_texto_limpo(resposta_fechamento.content)
    return {
        "pergunta": pergunta_cliente,
        "iteracoes": max_iter,
        "trilha_rag": trilha_rag,
        "resposta_final": texto_final
    }

print("Ferramenta '@tool consultar_manual_politicas_rag' e motor 'consultar_clara_rag_dia6' prontos!")

---
## Etapa 5: Bateria de Validação da Busca Semântica e Citação de Cláusulas

Submetemos o motor **Agentic RAG** às **4 consultas de validação** (`perguntas_rag_dia6`):
1. **Pergunta 1 (Código de Postagem Expirado):** Recuperação da **Seção 6 (6.1 e 6.2)** — validade inicial de **7 dias corridos** e direito a **1 renovação adicional de 5 dias corridos**.
2. **Pergunta 2 (Reembolso via Boleto Bancário):** Recuperação da **Seção 5 (e 5.1)** — conferência no CD em até **3 dias úteis** + depósito bancário em até **2 dias úteis** após aprovação (total de até **5 dias úteis**).
3. **Pergunta 3 (2ª Troca do Mesmo Pedido):** Recuperação da **Seção 2 (2.2)** — 1ª troca gratuita e cobrança de frete de envio/reenvio a partir da 2ª troca do mesmo pedido.
4. **Pergunta 4 (Gift Card Digital):** Recuperação da **Seção 4 (4.4)** — restrição de reembolso para *gift cards* e licenças digitais após o envio do código.

In [ ]:
perguntas_rag_dia6 = [
    "Recebi meu código de postagem dos Correios na semana passada, mas viajei e ele venceu sem eu ir aos Correios. Perdi meu direito de vez ou consigo renovar esse código? Por quantos dias?",
    "Eu paguei meu pedido via Boleto Bancário. Assim que a caixa devolvida chegar no Centro de Distribuição de vocês, quantos dias úteis no total (incluindo a conferência) vai demorar para o dinheiro cair na minha conta?",
    "Se eu já tiver feito a 1ª troca gratuita de tamanho de uma calça e o novo tamanho ainda não ficar bom, eu posso fazer uma 2ª troca do mesmo pedido? Quem paga o frete?",
    "Comprei um gift card digital para dar de presente, o código já foi enviado por e-mail, mas a pessoa não quer mais. Posso pedir reembolso?"
]

resultados_bateria_rag = []

for idx, pergunta in enumerate(perguntas_rag_dia6, 1):
    print("=" * 95)
    print(f"🧪 TESTE RAG #{idx} DE {len(perguntas_rag_dia6)}")
    print("=" * 95)
    resultado = consultar_clara_rag_dia6(pergunta, verbose=True)
    resultados_bateria_rag.append(resultado)
    print("\n")

---
### Auditoria dos Chunks Recuperados pelo ChromaDB em Cada Pergunta

Na célula abaixo, inspecionamos a trilha de auditoria (`trilha_rag`) de cada uma das 4 perguntas para comprovar quais seções do arquivo `politicas_troca.txt` foram recuperadas pela busca vetorial (`gemini-embedding-001`) no **ChromaDB**.

In [ ]:
print("=== AUDITORIA DE RECUPERAÇÃO VETORIAL (CHROMADB + GEMINI-EMBEDDING-001) ===\n")
for idx, res in enumerate(resultados_bateria_rag, 1):
    secoes = []
    queries = []
    for passo in res["trilha_rag"]:
        queries.append(passo["argumentos_busca"].get("duvida_politica", ""))
        for s in passo["secoes_recuperadas"]:
            if s not in secoes:
                secoes.append(s)
    print(f"Pergunta #{idx}: {res['pergunta'][:85]}...")
    print(f"  -> Query semântica gerada pela Clara : {queries}")
    print(f"  -> Seções Top-2 recuperadas (ChromaDB): {secoes}")
    print("-" * 95)

---
## Resumo da Implementação — Dia 6 Concluído

### Componentes construídos neste notebook:
- Carregamento e *chunking* semântico de **`politicas_troca.txt`** com `RecursiveCharacterTextSplitter` (`chunk_size=500`, `chunk_overlap=80`);
- Geração de Embeddings com **`models/gemini-embedding-001`** e indexação no **`ChromaDB`**;
- Encapsulamento do `retriever` (`k=2`) na ferramenta **`@tool consultar_manual_politicas_rag`** (**Agentic RAG**);
- Validação das respostas com citação explícita das seções oficiais do manual.

Compartilhe seu progresso nas redes sociais com a hashtag **#7DaysOfCode**!